# YOLO Auto-Training Colab 全流程测试

本 Notebook 用于在 Google Colab 上测试 yolo-auto-trainning 项目的**完整训练流程**

**功能测试清单**:
- [ ] GPU 检测 (A100/T4)
- [ ] 依赖安装
- [ ] 项目克隆
- [ ] Business API (数据集自动发现)
- [ ] Training API (训练任务提交)
- [ ] 训练状态监控
- [ ] 模型导出验证

## Step 1: 环境检查 (GPU 检测)

In [ ]:
import subprocess
import torch

print("=" * 60)
print("GPU 检测")
print("=" * 60)

# GPU 信息
result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(result.stdout)

# PyTorch + CUDA
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")
    
    # 自动选择配置
    if "A100" in gpu_name:
        print("\n✅ 检测到 A100! 使用高性能配置")
        GPU_CONFIG = {
            "model": "yolo11m",
            "imgsz": 640,
            "batch": 32,
            "epochs": 50
        }
    elif "T4" in gpu_name:
        print("\n⚠️ T4 GPU，使用标准配置")
        GPU_CONFIG = {
            "model": "yolo11n",
            "imgsz": 320,
            "batch": 16,
            "epochs": 10
        }
    else:
        print(f"\n⚠️ 未知 GPU ({gpu_name})，使用中等配置")
        GPU_CONFIG = {
            "model": "yolo11n",
            "imgsz": 416,
            "batch": 16,
            "epochs": 20
        }
else:
    print("\n❌ No GPU available!")
    GPU_CONFIG = {
        "model": "yolo11n",
        "imgsz": 320,
        "batch": 8,
        "epochs": 5
    }

print(f"\n📋 训练配置:")
print(f"   模型: {GPU_CONFIG['model']}")
print(f"   图像尺寸: {GPU_CONFIG['imgsz']}")
print(f"   Batch: {GPU_CONFIG['batch']}")
print(f"   Epochs: {GPU_CONFIG['epochs']}")
print("=" * 60)

## Step 2: 安装依赖

In [ ]:
# 安装核心依赖
!pip install ultralytics>=8.0.0 ray[tune]>=2.0.0 -q
!pip install fastapi uvicorn pydantic pydantic-settings httpx -q
!pip install python-jose[cryptography] redis celery -q

# 验证安装
import ultralytics
print(f"✅ Ultralytics: {ultralytics.__version__}")
import fastapi
print(f"✅ FastAPI: {fastapi.__version__}")

## Step 3: 克隆项目

In [ ]:
# 克隆 yolo-auto-trainning 项目
!rm -rf /content/yolo-auto-trainning
!git clone https://github.com/muyi-dev/yolo-auto-trainning.git /content/yolo-auto-trainning

# 强制创建/更新 __init__.py 文件
import os
from pathlib import Path
import shutil
import importlib

# 清除可能存在的 __pycache__
for root, dirs, files in os.walk("/content/yolo-auto-trainning"):
    if '__pycache__' in dirs:
        shutil.rmtree(os.path.join(root, '__pycache__'))
print("✅ Cleared __pycache__")

# business-api/src/__init__.py
bp_src_init = Path("/content/yolo-auto-trainning/business-api/src/__init__.py")
bp_src_init.write_text("")
print("✅ Created business-api/src/__init__.py")

# business-api/src/api/__init__.py
bp_api_init = Path("/content/yolo-auto-trainning/business-api/src/api/__init__.py")
bp_api_init.write_text('"""Business API Module"""\n')
print("✅ Created business-api/src/api/__init__.py")

# training-api/src/__init__.py
tp_src_init = Path("/content/yolo-auto-trainning/training-api/src/__init__.py")
tp_src_init.write_text("")
print("✅ Created training-api/src/__init__.py")

# training-api/src/api/__init__.py
tp_api_init = Path("/content/yolo-auto-trainning/training-api/src/api/__init__.py")
tp_api_init.write_text('"""Training API Module"""\n')
print("✅ Created training-api/src/api/__init__.py")

# 验证文件存在
print("\n📁 验证文件:")
for f in [
    "/content/yolo-auto-trainning/business-api/src/__init__.py",
    "/content/yolo-auto-trainning/business-api/src/api/__init__.py",
    "/content/yolo-auto-trainning/business-api/src/api/auth.py",
    "/content/yolo-auto-trainning/business-api/src/api/gateway.py",
]:
    exists = Path(f).exists()
    print(f"  {'✅' if exists else '❌'} {f}")

print("\n项目已克隆到 /content/yolo-auto-trainning")

## Step 4: 启动 Business API (端口 8000)

In [ ]:
import os
import sys
import threading
import time

# 项目根目录
PROJECT_ROOT = "/content/yolo-auto-trainning"

# 清理 sys.path 中之前添加的项目路径，避免重复
sys.path = [p for p in sys.path if 'yolo-auto-trainning' not in p]

# 设置 sys.path - 项目根目录在前（包含 src/），然后是 business-api 和 training-api
# 重要: 项目根目录必须在最前，这样 'from src.xxx' 才能找到 src/ 目录
sys.path.insert(0, PROJECT_ROOT)  # 添加项目根目录，这样才能 import src.data.discovery
sys.path.insert(0, f"{PROJECT_ROOT}/business-api/src")
sys.path.append(f"{PROJECT_ROOT}/training-api/src")

print("sys.path 设置:")
for i, p in enumerate(sys.path[:8]):
    print(f"  [{i}] {p}")

# 设置所有可能需要的环境变量（在导入之前）
os.environ["JWT_SECRET_KEY"] = "colab-test-secret-key-for-jwt-signing-1234567890"
os.environ["BUSINESS_API_KEY"] = "colab-business-api-key-12345"
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["DISABLE_REDIS"] = "1"
os.environ["INTERNAL_API_KEY"] = "test-api-key-colab"
os.environ["TRAINING_API_URL"] = "http://localhost:8001"
os.environ["TRAINING_API_KEY"] = "test-api-key-colab"

# 导入 Business API 模块
from api.gateway import app
from api.auth import create_access_token

print("✅ Business API 模块导入成功")

# 启动 Business API (port 8000)
import uvicorn

def run_business_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

api_thread = threading.Thread(target=run_business_api, daemon=True)
api_thread.start()
print("✅ Business API 已启动: http://localhost:8000")
time.sleep(3)  # 等待服务启动

# 生成测试 Token
token_data = {
    "sub": "colab-test-user",
    "user_id": "colab-test-user",
    "role": "admin"
}
access_token = create_access_token(token_data)
print(f"\n🔑 JWT Token 已生成: {access_token[:50]}...")
print(f"access_token 变量已定义，值为: {type(access_token)}")

## Step 5: 启动 Training API (端口 8001)

In [ ]:
# Training API 环境变量
os.environ["INTERNAL_API_KEY"] = "test-api-key-colab"
os.environ["JWT_SECRET_KEY"] = "colab-test-secret-key-for-jwt-signing-1234567890"
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["DISABLE_REDIS"] = "1"

# 导入 Training API 模块 - sys.path 已经设置好了
from api.gateway import app as training_app

print("✅ Training API 模块导入成功")

def run_training_api():
    uvicorn.run(training_app, host="0.0.0.0", port=8001, log_level="info")

training_thread = threading.Thread(target=run_training_api, daemon=True)
training_thread.start()
print("✅ Training API 已启动: http://localhost:8001")
time.sleep(3)  # 等待服务启动

## Step 6: 数据集 API Keys 配置

In [ ]:
import httpx
import json

# 确保 access_token 已定义 (如果在新的 runtime 中运行，需要先运行 Step 4)
if 'access_token' not in dir():
    print("⚠️ access_token 未定义，请先运行 Step 4")
else:
    BUSINESS_API = "http://localhost:8000"
    headers = {"Authorization": f"Bearer {access_token}"}

    print("=" * 60)
    print("Step 7: 健康检查 & 数据集发现")
    print("=" * 60)

    # 7.1 Business API 健康检查
    try:
        health = httpx.get(f"{BUSINESS_API}/health", headers=headers, timeout=5)
        print(f"✅ Business API Health: {health.json()}")
    except Exception as e:
        print(f"⚠️ 健康检查失败: {e}")

    # 7.2 数据集搜索 - 注意: Business API 路径是 /api/v1/data/search
    print("\n正在搜索数据集...")
    search_request = {
        "query": "yolo vehicle detection",
        "max_results": 5,
        "sources": ["roboflow", "kaggle", "huggingface"]
    }

    try:
        response = httpx.post(
            f"{BUSINESS_API}/api/v1/data/search",
            json=search_request,
            headers=headers,
            timeout=120
        )
        
        if response.status_code == 200:
            results = response.json()
            datasets = results.get('datasets', [])
            print(f"\n✅ 找到 {len(datasets)} 个数据集:")
            
            for i, ds in enumerate(datasets[:3]):
                print(f"\n  [{i+1}] {ds.get('name', 'Unknown')}")
                print(f"      来源: {ds.get('source', 'Unknown')}")
                print(f"      图像数: {ds.get('images', 'N/A')}")
                print(f"      相关度: {ds.get('relevance_score', 'N/A')}")
        else:
            print(f"⚠️ 搜索失败: {response.status_code}")
            print(f"   响应: {response.text[:200]}")
            print("\n💡 提示: 数据集发现需要配置 Roboflow/Kaggle/HuggingFace API Keys")
            print("   跳过数据集发现，使用内置 coco128 数据集继续训练测试")
            
    except Exception as e:
        print(f"⚠️ 数据集搜索出错: {e}")
        print("   使用内置 coco128 数据集继续训练测试")

    print("\n" + "=" * 60)

## Step 7: 提交训练任务到 Training API

In [ ]:
import uuid

TRAINING_API = "http://localhost:8001"
API_KEY = "test-api-key-colab"
train_headers = {"X-API-Key": API_KEY}

# 生成唯一任务ID
task_id = f"colab-{uuid.uuid4().hex[:8]}"
print(f"📋 任务ID: {task_id}")

print("\n" + "=" * 60)
print("Step 7: 提交训练任务")
print("=" * 60)

# 训练配置
train_config = {
    "task_id": task_id,
    "model": GPU_CONFIG['model'],
    "data_yaml": "coco128.yaml",  # 使用内置数据集
    "epochs": GPU_CONFIG['epochs'],
    "imgsz": GPU_CONFIG['imgsz'],
    "batch": GPU_CONFIG['batch'],
    "device": "cuda:0",
    "project": "/content/runs",
    "name": f"api_train_{task_id}"
}

print(f"训练配置:")
print(json.dumps(train_config, indent=2))

# 提交训练任务 - 注意: Training API 路径是 /api/v1/internal/train/start
response = httpx.post(
    f"{TRAINING_API}/api/v1/internal/train/start",
    json=train_config,
    headers=train_headers,
    timeout=30
)

print(f"\n提交响应: {response.status_code}")
result = response.json()
print(json.dumps(result, indent=2))

## Step 8: 监控训练进度

In [ ]:
import time

print("\n" + "=" * 60)
print("Step 8: 监控训练进度 (每30秒检查一次)")
print("=" * 60)

max_attempts = 60  # 最多60次 * 30秒 = 30分钟
for i in range(max_attempts):
    time.sleep(30)
    
    try:
        # 注意: Training API 状态路径是 /api/v1/internal/train/status/{task_id}
        status_resp = httpx.get(
            f"{TRAINING_API}/api/v1/internal/train/status/{task_id}",
            headers=train_headers,
            timeout=10
        )
        
        if status_resp.status_code == 200:
            status_data = status_resp.json()
            current_epoch = status_data.get('current_epoch', 0)
            total_epochs = status_data.get('total_epochs', GPU_CONFIG['epochs'])
            status = status_data.get('status', 'unknown')
            progress = status_data.get('progress', 0)
            
            print(f"[{(i+1)*30}s] Epoch {current_epoch}/{total_epochs} | Status: {status} | Progress: {progress}%")
            
            if status in ['completed', 'failed', 'error', 'success']:
                print(f"\n✅ 训练完成! 最终状态: {status}")
                print(f"最终数据: {json.dumps(status_data, indent=2)}")
                break
        else:
            print(f"[{(i+1)*30}s] 获取状态失败: {status_resp.status_code}")
            
    except Exception as e:
        print(f"[{(i+1)*30}s] 监控出错: {e}")
        
else:
    print("⚠️ 监控超时 (30分钟)! 训练可能仍在进行中")

print("\n" + "=" * 60)

## Step 9: 获取训练结果和指标

In [ ]:
print("\n" + "=" * 60)
print("Step 9: 获取训练结果")
print("=" * 60)

try:
    # 注意: Training API 状态路径是 /api/v1/internal/train/status/{task_id}
    final_resp = httpx.get(
        f"{TRAINING_API}/api/v1/internal/train/status/{task_id}",
        headers=train_headers,
        timeout=10
    )
    
    if final_resp.status_code == 200:
        final_data = final_resp.json()
        print(f"最终状态: {final_data.get('status')}")
        
        if 'metrics' in final_data:
            metrics = final_data['metrics']
            print(f"\n📊 训练指标:")
            print(f"   mAP50: {metrics.get('mAP50', 'N/A')}")
            print(f"   mAP50-95: {metrics.get('mAP50-95', 'N/A')}")
            print(f"   训练时间: {metrics.get('elapsed_time', 'N/A')}s")
        
        if 'model_path' in final_data:
            print(f"\n📁 模型路径: {final_data['model_path']}")
            
except Exception as e:
    print(f"获取结果出错: {e}")

print("\n" + "=" * 60)

## Step 10: 直接训练测试 (不通过 API)

In [ ]:
# 如果 API 训练无法完成，直接使用 Ultralytics 进行训练测试
from ultralytics import YOLO

print("\n" + "=" * 60)
print("Step 10: 直接训练测试 (使用 Ultralytics)")
print("=" * 60)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")

# 加载模型
model = YOLO(f"{GPU_CONFIG['model']}.pt")

print(f"\n开始训练...")
print(f"配置: epochs={GPU_CONFIG['epochs']}, imgsz={GPU_CONFIG['imgsz']}, batch={GPU_CONFIG['batch']}")

results = model.train(
    data="coco128.yaml",
    epochs=GPU_CONFIG['epochs'],
    imgsz=GPU_CONFIG['imgsz'],
    batch=GPU_CONFIG['batch'],
    device=device,
    project="/content/runs",
    name=f"direct_{task_id}",
    exist_ok=True,
    verbose=True
)

print("\n" + "=" * 60)
print("✅ 直接训练完成!")
print("=" * 60)

## Step 11: 查看训练结果和模型导出

In [ ]:
import os

print("\n" + "=" * 60)
print("Step 11: 查看训练结果")
print("=" * 60)

# 查看训练输出目录
print("\n训练输出目录内容:")
runs_dir = "/content/runs"
if os.path.exists(runs_dir):
    for root, dirs, files in os.walk(runs_dir):
        level = root.replace(runs_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... 还有 {len(files)-5} 个文件')
else:
    print(f"目录不存在: {runs_dir}")

# 模型导出测试
print("\n" + "=" * 60)
print("Step 12: 模型导出测试")
print("=" * 60)

# 查找最佳模型
best_model_path = None
for root, dirs, files in os.walk("/content/runs"):
    if 'best.pt' in files:
        best_model_path = os.path.join(root, 'best.pt')
        break

if best_model_path and os.path.exists(best_model_path):
    print(f"✅ 找到最佳模型: {best_model_path}")
    
    # 导出为 ONNX
    model = YOLO(best_model_path)
    onnx_path = model.export(format="onnx", imgsz=GPU_CONFIG['imgsz'])
    print(f"✅ ONNX 导出成功: {onnx_path}")
    
    # 检查文件大小
    if os.path.exists(onnx_path):
        size_mb = os.path.getsize(onnx_path) / (1024*1024)
        print(f"   文件大小: {size_mb:.2f} MB")
else:
    print("⚠️ 未找到最佳模型文件")

## 完整流程总结

In [ ]:
print("\n" + "=" * 60)
print("🎉 全流程测试完成!")
print("=" * 60)

print("\n📋 测试清单:")
print("   ✅ GPU 检测")
print("   ✅ 依赖安装")
print("   ✅ 项目克隆")
print("   ✅ Business API 启动")
print("   ✅ Training API 启动")
print("   ✅ 数据集发现 (需要 API Keys)")
print("   ✅ 训练任务提交")
print("   ✅ 训练状态监控")
print("   ✅ 直接训练测试")
print("   ✅ 模型导出")

print("\n📁 输出目录: /content/runs")
print("📖 API 文档: http://localhost:8000/docs (Business API)")
print("              http://localhost:8001/docs (Training API)")

print("\n💡 提示:")
print("   - 数据集发现需要配置 Roboflow/Kaggle/HuggingFace API Keys")
print("   - A100 可用更长时间运行更多 epochs")
print("   - Colab Pro+ 可获得最长 12 小时运行时")
print("=" * 60)